In [52]:
import sqlite3
from sqlite3 import Error
from sqlalchemy import create_engine, inspect, text
import pandas as pd


In [ ]:
def create_connection(db_file):
    """ create a database connection to the SQLite database
        specified by the db_file
    :param db_file: database file
    :return: Connection object or None
    """
    conn = None
    try:
        conn = sqlite3.connect(db_file)
        print(sqlite3.version)
    except Error as e:
        print(e)
 
    return conn

 
def select_all_tasks(conn):
    """
    Query all rows in the tasks table
    :param conn: the Connection object
    :return:
    """
    cur = conn.cursor()
    
    query1 = """
        SELECT *
        FROM FACILITIES
        """
    cur.execute(query1)
 
    rows = cur.fetchall()
 
    for row in rows:
        print(row)


def main():
    database = "sqlite_db_pythonsqlite.db"
 
    # create a database connection
    conn = create_connection(database)
    with conn: 
        print("2. Query all tasks")
        select_all_tasks(conn)
 
 
if __name__ == '__main__':
    main()

# Exploring the Database with SQLAlchemy
Let's explore the database before writing any queries. We'll use SQLAlchemy and pandas for this.

In [45]:
# 1. List all tables in the database

# Create a connection to the SQLite database using SQLAlchemy
engine = create_engine("sqlite:///sqlite_db_pythonsqlite.db")
inspector = inspect(engine)

# Get the names of all tables in the database
tables = inspector.get_table_names()
print("Tables in the database:")

# Print each table name
for table in tables:
    print(table)

Tables in the database:
Bookings
Facilities
Members


In [46]:
# 2. Show columns and schema for each table

# For each table, print its name and list all columns with their data types
for table in tables:
    print(f"\nTable: {table}")

    # Get column information for the current table. columns is a list of dictionaries. Each dictionary contains details about a column in the table.
    columns = inspector.get_columns(table)

    # Print each column's name and type
    for col in columns:
        print(f"  {col['name']} ({col['type']})")


Table: Bookings
  bookid (INTEGER)
  facid (INTEGER)
  memid (INTEGER)
  starttime (VARCHAR(19))
  slots (INTEGER)

Table: Facilities
  facid (INTEGER)
  name (VARCHAR(15))
  membercost (DECIMAL(2, 1))
  guestcost (DECIMAL(3, 1))
  initialoutlay (INTEGER)
  monthlymaintenance (INTEGER)

Table: Members
  memid (INTEGER)
  surname (VARCHAR(17))
  firstname (VARCHAR(9))
  address (VARCHAR(39))
  zipcode (INTEGER)
  telephone (VARCHAR(14))
  recommendedby (VARCHAR(2))
  joindate (VARCHAR(19))


/var/folders/xk/1_94y7fn37sf7kbch2g77hlm0000gp/T/ipykernel_29119/1204765523.py:8: SAWarning: Could not instantiate type <class 'sqlalchemy.sql.sqltypes.INTEGER'> with reflected arguments ['4']; using no arguments.
  columns = inspector.get_columns(table)
/var/folders/xk/1_94y7fn37sf7kbch2g77hlm0000gp/T/ipykernel_29119/1204765523.py:8: SAWarning: Could not instantiate type <class 'sqlalchemy.sql.sqltypes.INTEGER'> with reflected arguments ['1']; using no arguments.
  columns = inspector.get_columns(table)
/var/folders/xk/1_94y7fn37sf7kbch2g77hlm0000gp/T/ipykernel_29119/1204765523.py:8: SAWarning: Could not instantiate type <class 'sqlalchemy.sql.sqltypes.INTEGER'> with reflected arguments ['2']; using no arguments.
  columns = inspector.get_columns(table)
/var/folders/xk/1_94y7fn37sf7kbch2g77hlm0000gp/T/ipykernel_29119/1204765523.py:8: SAWarning: Could not instantiate type <class 'sqlalchemy.sql.sqltypes.INTEGER'> with reflected arguments ['5']; using no arguments.
  columns = inspector

In [47]:
# 3. Preview the first few rows of each table
# For each table, use pandas to read and display the first 5 rows
for table in tables:
    print(f"\nPreview of {table}:")
    # Read the first 5 rows from the table into a pandas DataFrame
    df = pd.read_sql(f"SELECT * FROM {table} LIMIT 5", engine)
    # Print the DataFrame to see sample data
    print(df)


Preview of Bookings:
   bookid  facid  memid            starttime  slots
0       0      3      1  2012-07-03 11:00:00      2
1       1      4      1  2012-07-03 08:00:00      2
2       2      6      0  2012-07-03 18:00:00      2
3       3      7      1  2012-07-03 19:00:00      2
4       4      8      1  2012-07-03 10:00:00      1

Preview of Facilities:
   facid             name  membercost  guestcost  initialoutlay  \
0      0   Tennis Court 1         5.0       25.0          10000   
1      1   Tennis Court 2         5.0       25.0           8000   
2      2  Badminton Court         0.0       15.5           4000   
3      3     Table Tennis         0.0        5.0            320   
4      4   Massage Room 1         9.9       80.0           4000   

   monthlymaintenance  
0                 200  
1                 200  
2                  50  
3                  10  
4                3000  

Preview of Members:
   memid   surname firstname                       address  zipcode  \
0  

In [48]:
# 4. Summarize columns: count, unique values, and missing values for each table
# For each table, use pandas to show info, unique values, and missing values
for table in tables:
    print(f"\nSummary statistics for {table}:")
    # Read the entire table into a pandas DataFrame
    df = pd.read_sql(f"SELECT * FROM {table}", engine)
    # Print info about columns, data types, and non-null counts
    print(df.info())
    print("\nUnique values per column:")
    # Print the number of unique values for each column
    for col in df.columns:
        print(f"  {col}: {df[col].nunique()} unique values")
    print("\nMissing values per column:")
    # Print the number of missing (null) values for each column
    print(df.isnull().sum())


Summary statistics for Bookings:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4043 entries, 0 to 4042
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   bookid     4043 non-null   int64 
 1   facid      4043 non-null   int64 
 2   memid      4043 non-null   int64 
 3   starttime  4043 non-null   object
 4   slots      4043 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 158.1+ KB
None

Unique values per column:
  bookid: 4043 unique values
  facid: 9 unique values
  memid: 30 unique values
  starttime: 1814 unique values
  slots: 9 unique values

Missing values per column:
bookid       0
facid        0
memid        0
starttime    0
slots        0
dtype: int64

Summary statistics for Facilities:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   facid      

# Questions

**Q1: Some of the facilities charge a fee to members, but some do not.**
- Write a SQL query to produce a list of the names of the facilities that do.


In [80]:
df = pd.read_sql(
    """
    SELECT 
        name, 
        membercost 
    FROM Facilities 
    WHERE membercost > 0
    """, 
    engine
    )
print(df)

             name  membercost
0  Tennis Court 1         5.0
1  Tennis Court 2         5.0
2  Massage Room 1         9.9
3  Massage Room 2         9.9
4    Squash Court         3.5


**Q2: How many facilities do not charge a fee to members?**

In [79]:
df = pd.read_sql(
    """
    SELECT 
        name, 
        membercost 
    FROM Facilities
    WHERE membercost = 0
    """,
    engine
    )
print(df)

              name  membercost
0  Badminton Court           0
1     Table Tennis           0
2    Snooker Table           0
3       Pool Table           0


**Q3: Write an SQL query to show a list of facilities that charge a fee to members,
where the fee is less than 20% of the facility's monthly maintenance cost.**

Return the facid, facility name, member cost, and monthly maintenance of the
facilities in question.

In [ ]:
df = pd.read_sql(
    """
    SELECT 
        facid, 
        name, 
        membercost, 
        monthlymaintenance 
    FROM Facilities 
    WHERE membercost > 0 
        AND (membercost / monthlymaintenance < .2)
    """, 
    engine
    )
print(df)

   facid            name  membercost  monthlymaintenance
0      0  Tennis Court 1         5.0                 200
1      1  Tennis Court 2         5.0                 200
2      4  Massage Room 1         9.9                3000
3      5  Massage Room 2         9.9                3000
4      6    Squash Court         3.5                  80


**Q4: Write an SQL query to retrieve the details of facilities with ID 1 and 5.
Try writing the query without using the OR operator.**

In [77]:
df = pd.read_sql(
    """
    SELECT * 
    FROM Facilities 
    WHERE facid IN (1, 5)
    """, 
    engine
    )
print(df)

   facid            name  membercost  guestcost  initialoutlay  \
0      1  Tennis Court 2         5.0         25           8000   
1      5  Massage Room 2         9.9         80           4000   

   monthlymaintenance  
0                 200  
1                3000  


**Q5: Produce a list of facilities, with each labelled as 'cheap' or 'expensive', depending on if their monthly maintenance cost is more than $100. Return the name and monthly maintenance of the facilities in question.**

In [75]:
df = pd.read_sql(
    """
    SELECT name,
           monthlymaintenance,
           CASE WHEN monthlymaintenance > 100 THEN 'Expensive'
                ELSE 'cheap' END AS cost_status
    FROM Facilities
    """,
    engine
)
print(df)

              name  monthlymaintenance cost_status
0   Tennis Court 1                 200   Expensive
1   Tennis Court 2                 200   Expensive
2  Badminton Court                  50       cheap
3     Table Tennis                  10       cheap
4   Massage Room 1                3000   Expensive
5   Massage Room 2                3000   Expensive
6     Squash Court                  80       cheap
7    Snooker Table                  15       cheap
8       Pool Table                  15       cheap


**Q6: You'd like to get the first and last name of the last member(s)
who signed up. Try not to use the LIMIT clause for your solution.**

In [94]:
df = pd.read_sql(
    """
    SELECT
        firstname,
        surname,
        joindate
    FROM Members
    WHERE joindate = (SELECT MAX(joindate) FROM Members)
    """,
    engine
)
print(df)

  firstname surname             joindate
0    Darren   Smith  2012-09-26 18:08:45


**Q7: Produce a list of all members who have used a tennis court.**
- Include in your output the name of the court, and 
- name of the member formatted as a single column. 
- Ensure no duplicate data, and 
- order by the member name.

In [97]:
df_facilities_list = pd.read_sql(
    """
    SELECT
        facid,
        name
    FROM Facilities
    """,
    engine
)
print(df_facilities_list)


   facid             name
0      0   Tennis Court 1
1      1   Tennis Court 2
2      2  Badminton Court
3      3     Table Tennis
4      4   Massage Room 1
5      5   Massage Room 2
6      6     Squash Court
7      7    Snooker Table
8      8       Pool Table


In [113]:
df = pd.read_sql(
   """
   SELECT DISTINCT
      Members.firstname || ' ' || Members.surname AS member_name,
      Facilities.name AS facility_name
   FROM Members
   INNER JOIN Bookings USING(memid)
   INNER JOIN Facilities USING(facid)
   WHERE Facilities.name LIKE 'Tennis Court%'
   ORDER BY member_name, facility_name
   """,
   engine
)
print(df.head())


    member_name   facility_name
0    Anne Baker  Tennis Court 1
1    Anne Baker  Tennis Court 2
2  Burton Tracy  Tennis Court 1
3  Burton Tracy  Tennis Court 2
4  Charles Owen  Tennis Court 1


**Q8: Produce a list of bookings on the day of 2012-09-14 which
will cost the member (or guest) more than $30. Remember that guests have
different costs to members (the listed costs are per half-hour 'slot'), and
the guest user's ID is always 0. Include in your output the name of the
facility, the name of the member formatted as a single column, and the cost.
Order by descending cost, and do not use any subqueries.**



In [1]:
df = pd.read_sql(
   """
   SELECT
      Members.firstname || ' ' || Members.surname AS member_name,
      Facilities.name AS facility_name,
      Facilities.membercost,
      Facilities.guestcost,
      DATE(Bookings.starttime) AS booking_date, 
      CASE 
         WHEN Members.member_name = 'GUEST GUEST' 
         THEN Facilities.guestcost
         ELSE Facilities.membercost END AS cost
   FROM Members
   INNER JOIN Bookings USING(memid)
   INNER JOIN Facilities USING(facid)
   WHERE booking_date = '2012-09-14'
      AND cost > 30
   ORDER BY membercost, guestcost
   """,
   engine
)
print(df.head())

NameError: name 'pd' is not defined

**Q9: This time, produce the same result as in Q8, but using a subquery.**